# Financial Transaction Fraud Detection

## Project Overview

This project explores a large-scale financial transaction dataset with the goal of identifying fraudulent activity patterns. The dataset contains anonymized credit card transactions and is highly imbalanced, making fraud detection a realistic and challenging classification problem.

In this notebook, I set up a local SQL database (SQLite), load the dataset into a structured table, and perform initial validation queries to confirm data integrity.

---

## Objectives

- Load raw transaction data into a SQL database
- Create a structured table for analysis
- Validate successful ingestion
- Run initial exploratory SQL queries

## Dataset Description

The dataset used in this project contains credit card transactions made by European cardholders.

Each row represents a single transaction and includes:

- **Time**: Seconds elapsed between this transaction and the first transaction in the dataset
- **Amount**: Transaction amount
- **V1–V28**: Anonymized features resulting from PCA transformation
- **Class**: Target variable (0 = legitimate transaction, 1 = fraud)

---

## Key Challenge

This dataset is highly imbalanced:
- Fraud cases represent a very small fraction of total transactions (~0.017%)

This makes accuracy a poor evaluation metric and motivates the need for careful analysis.

## Methodology

The data pipeline in this notebook follows three steps:

1. **Data Ingestion**
   - Load raw CSV data using pandas

2. **Database Creation**
   - Create a local SQLite database
   - Store transaction data in a structured SQL table

3. **Validation Queries**
   - Run SQL queries to confirm successful ingestion
   - Perform initial aggregations to understand dataset structure

In [2]:
import pandas as pd
import sqlite3

df = pd.read_csv("../data/creditcard.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [32]:
conn = sqlite3.connect('../fraud.db')
df.to_sql("transactions", conn, if_exists="replace", index=False)

284807

## Why SQL?

Although the dataset is small enough to fit in memory, SQL is used to simulate a production-style analytics workflow.

In real-world financial systems:
- Transaction data is stored in relational databases
- Analysts use SQL to query and aggregate data
- Downstream ML pipelines often depend on SQL-extracted features

This project mirrors that workflow to demonstrate practical data engineering and analytics skills.

In [33]:
pd.read_sql("""
    SELECT COUNT(*) AS transaction_count
    FROM transactions
""", conn)

,transaction_count
0,284807


In [36]:
pd.read_sql("""
    SELECT Class, COUNT(*) AS count
    FROM transactions
    GROUP BY Class
""",conn)

,Class,count
0,0,284315
1,1,492


In [37]:
pd.read_sql("""
    SELECT 
    SUM(CASE WHEN Class=1 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS fraud_rate
    FROM transactions
""", conn)

,fraud_rate
0,0.001727


In [38]:
pd.read_sql("""
    SELECT Class, AVG(Amount) AS avg_amount
    FROM transactions
    GROUP BY Class
""",conn)

,Class,avg_amount
0,0,88.291022
1,1,122.211321


In [40]:
pd.read_sql("""
    SELECT Time, Amount
    FROM transactions
    WHERE Class=1
    ORDER BY Amount DESC
    LIMIT 10
""",conn)

,Time,Amount
0,122608.0,2125.87
1,9064.0,1809.68
2,154278.0,1504.93
3,62467.0,1402.16
4,59011.0,1389.56
5,65385.0,1354.25
6,133184.0,1335.00
7,18088.0,1218.89
8,154309.0,1096.99
9,147501.0,996.27


## Initial Validation Results

After loading the dataset into SQLite, I performed basic validation queries:

- Total number of transactions
- Class distribution (fraud vs non-fraud)
- Basic transaction statistics

These checks ensure that the data was correctly ingested and is ready for further analysis.

## Next Steps

In the next notebook, I will perform exploratory data analysis (EDA) to:

- Understand transaction distributions
- Compare fraud vs non-fraud behavior
- Identify early patterns that may indicate fraudulent activity

This will guide feature selection and model design in later stages.